# Test the H2O Endpoint Locally

Start the Azure ML inference server with the checked-in H2O scorer and reference bundle, call health and scoring routes, verify golden parity, and stop the local server.

**Source:** Adapted from this repository's `notebooks/h2o_mojo/03_test_local_online_endpoint.ipynb`.

In [ ]:
from pathlib import Path
import json
import os
import platform
import shutil
import socket
import subprocess
import sys
import time

import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / ".env.example").is_file() and (candidate / "pipelines").is_dir():
        WORKSHOP_ROOT = candidate
        break
else:
    raise FileNotFoundError("Run this notebook from inside the workshop folder")
load_dotenv(WORKSHOP_ROOT / ".env", override=True)

bundle_value = Path(os.environ["H2O_BUNDLE_DIR"])
BUNDLE_DIR = bundle_value if bundle_value.is_absolute() else WORKSHOP_ROOT / bundle_value
SCORE_PATH = WORKSHOP_ROOT / "src/h2o/online/score.py"
LOCAL_DIR = WORKSHOP_ROOT / "outputs/h2o_local_endpoint"
LOCAL_DIR.mkdir(parents=True, exist_ok=True)
REQUEST_PATH = LOCAL_DIR / "request.json"
LOG_PATH = LOCAL_DIR / "server.log"
PORT = 5001

manifest = json.loads((BUNDLE_DIR / "model_manifest.json").read_text(encoding="utf-8"))
golden_input = pd.read_csv(BUNDLE_DIR / "golden_input.csv")
golden_expected = pd.read_csv(BUNDLE_DIR / "golden_expected.csv")
request_payload = {
    "input_data": {
        "columns": manifest["features"],
        "data": golden_input[manifest["features"]].values.tolist(),
    }
}
REQUEST_PATH.write_text(json.dumps(request_payload, indent=2), encoding="utf-8")

inference_server = shutil.which("azmlinfsrv")
if not inference_server:
    candidate = Path(sys.executable).with_name("azmlinfsrv.exe" if platform.system() == "Windows" else "azmlinfsrv")
    if candidate.is_file():
        inference_server = str(candidate)
if not inference_server:
    raise FileNotFoundError("azmlinfsrv was not found in the selected Python environment")

def port_is_open(port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as connection:
        connection.settimeout(0.25)
        return connection.connect_ex(("127.0.0.1", port)) == 0

def stop_server(process, log_handle) -> None:
    if process is not None and process.poll() is None:
        if platform.system() == "Windows":
            subprocess.run(["taskkill", "/PID", str(process.pid), "/T", "/F"], capture_output=True, check=False)
        else:
            process.terminate()
        try:
            process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            process.kill()
    if log_handle is not None and not log_handle.closed:
        log_handle.close()

if port_is_open(PORT) or port_is_open(54321):
    raise RuntimeError("Ports 5001 and 54321 must be free before startup")

server_process = None
server_log_handle = LOG_PATH.open("w", encoding="utf-8")
try:
    server_environment = os.environ.copy()
    for inherited_key in ("AZUREML_ENTRY_SCRIPT", "AZUREML_MODEL_DIR", "WORKSPACE_NAME"):
        server_environment.pop(inherited_key, None)
    server_environment["PYTHONUNBUFFERED"] = "1"
    server_environment["WORKER_COUNT"] = "1"
    command = [
        inference_server,
        "--entry_script", str(SCORE_PATH),
        "--model_dir", str(BUNDLE_DIR),
        "--port", str(PORT),
        "--worker_count", "1",
    ]
    server_process = subprocess.Popen(command, cwd=LOCAL_DIR, env=server_environment, stdout=server_log_handle, stderr=subprocess.STDOUT)

    health_url = f"http://127.0.0.1:{PORT}/"
    deadline = time.monotonic() + 90
    while time.monotonic() < deadline:
        if server_process.poll() is not None:
            server_log_handle.flush()
            raise RuntimeError(LOG_PATH.read_text(encoding="utf-8", errors="replace"))
        try:
            health_response = requests.get(health_url, timeout=1)
            if health_response.status_code == 200:
                break
        except requests.RequestException:
            pass
        time.sleep(1)
    else:
        raise TimeoutError("The local inference server did not become healthy")

    score_response = requests.post(f"http://127.0.0.1:{PORT}/score", json=request_payload, timeout=30)
    score_response.raise_for_status()
    body = score_response.json()
    predictions = np.asarray(body["predictions"], dtype=float)
    np.testing.assert_allclose(golden_expected["predict"], predictions, rtol=1e-6, atol=1e-6)
    print(f"Health: {health_response.status_code}")
    print(f"Scored rows: {len(predictions)}")
    print(f"Model: {body['model_name']}:{body['model_version']}")
finally:
    stop_server(server_process, server_log_handle)

if port_is_open(PORT):
    raise RuntimeError("The local inference server did not stop cleanly")
print("Local endpoint parity passed and the server stopped cleanly.")

## Expected Result

The local server becomes healthy, scores all golden rows within tolerance, and releases ports 5001 and 54321 during cleanup.

Next: `04_deploy_reference_endpoint.ipynb`.